# Chuẩn hóa dữ liệu — Phase 2 (lượt chạy lại)

Chỉ chuẩn hóa kiểu/định dạng, giữ nguyên mọi dòng, mã, nhãn theo H1 đã chốt. Mỗi cell một việc, giải thích ngay dưới.

In [1]:
import pandas as pd, openpyxl
print('pandas:', pd.__version__, '| openpyxl:', openpyxl.__version__)
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'Nile_Sales_YTD.xlsx').exists()))
DATA = ROOT / 'data' / 'Nile_Sales_YTD.xlsx'
OUT = ROOT / 'data' / 'processed'
CLEAN = OUT / 'nile_sales_clean.csv'
FLAGS = OUT / 'nile_sales_review_flags.csv'
import hashlib
sha_dau = hashlib.sha256(DATA.read_bytes()).hexdigest()
assert sha_dau == 'fae7a6955da3f3166ef4766d26acc39b8fed7b8c763e077a83c015a6d4fc0bd8', 'DUNG: hash nguon doi'
print('SHA khớp bản hiện hành.')
COLS = ['Sales_Date','Delivery_Date','SKU_Code','Qty','Unit_Price','Line_Amount','From_Location','To_Location','Channel','Sub_Channel','Delivery_Type','Courier','City_Tier','Category','Order_ID','Customer_ID','Payment_Method','Is_Returned']
SHEETS = {'Nile_Sales_2025-07': 1722, 'Nile_Sales_2025-06': 1597, 'Nile_Sales_2025-05': 1473, 'Nile_Sales_2025-04': 1376, 'Nile_Sales_2025-03': 1276, 'Nile_Sales_2025-02': 1167, 'Nile_Sales_2025-01': 1094}
TRACE = ['__source_file__', '__sheet__', '__excel_row__']


pandas: 2.2.3 | openpyxl: 3.1.5
SHA khớp bản hiện hành.


**Giải thích:** hash nguồn khớp bản hiện hành thì làm tiếp; lệch thì dừng, không tự thay expected.

In [2]:
wb_goc = openpyxl.load_workbook(DATA, read_only=True, data_only=True)
assert sorted(wb_goc.sheetnames) == sorted(SHEETS)
for ten_sheet in wb_goc.sheetnames:
    cac_dong = list(wb_goc[ten_sheet].iter_rows(values_only=True))
    assert list(cac_dong[0]) == COLS
    dem = sum(1 for dong_o in cac_dong[1:] if any(o_goc is not None for o_goc in dong_o))
    assert dem == SHEETS[ten_sheet], f'DUNG sheet {ten_sheet}'
wb_goc.close()
print('7 sheet đủ dòng/schema 18 cột.')
cac_bang = []
for ten_sheet in SHEETS:
    bang = pd.read_excel(DATA, sheet_name=ten_sheet, engine='openpyxl')
    bang['__source_file__'] = 'data/Nile_Sales_YTD.xlsx'
    bang['__sheet__'] = ten_sheet
    bang['__excel_row__'] = bang.index + 2
    cac_bang.append(bang)
nguon = pd.concat(cac_bang, ignore_index=True)
ma_truy = nguon['__source_file__'] + '|' + nguon['__sheet__'] + '|' + nguon['__excel_row__'].astype(str)
assert len(nguon) == 9705 and ma_truy.is_unique
print('Gộp 9.705 dòng; mã truy nguồn duy nhất.')


7 sheet đủ dòng/schema 18 cột.


Gộp 9.705 dòng; mã truy nguồn duy nhất.


**Giải thích:** đọc trực tiếp Excel, giữ địa chỉ ô gốc cho từng dòng. Không dùng CSV cũ.

In [3]:
ngay_ban = pd.to_datetime(nguon.Sales_Date, errors='coerce')
ngay_giao = pd.to_datetime(nguon.Delivery_Date, errors='coerce')
assert int(ngay_ban.isna().sum()) == 0 and int(ngay_giao.isna().sum()) == 0
gio_le = ((ngay_ban.dt.hour != 0) | (ngay_ban.dt.minute != 0) | (ngay_ban.dt.second != 0) | (ngay_ban.dt.microsecond != 0)).sum()
gio_le += ((ngay_giao.dt.hour != 0) | (ngay_giao.dt.minute != 0) | (ngay_giao.dt.second != 0) | (ngay_giao.dt.microsecond != 0)).sum()
assert int(gio_le) == 0
print('Ngày parse đủ, không NaT; giờ/phút/giây/micro đều 0 → ISO ngày không mất thông tin.')
for ten_cot in ['Qty', 'Unit_Price', 'Line_Amount', 'Is_Returned']:
    cot = pd.to_numeric(nguon[ten_cot], errors='coerce')
    assert int(cot.isna().sum()) == 0, f'DUNG null {ten_cot}'
    assert bool((cot == cot.astype('int64')).all()), f'DUNG phan thap phan {ten_cot}'
    assert nguon[ten_cot].dtype == 'int64', f'DUNG kieu {ten_cot}'
print('4 cột số/cờ: đúng int64, không null, phần thập phân bằng 0.')
assert int(nguon[COLS].isna().sum().sum()) == 0
print('Chuỗi/mã giữ nguyên giá trị, không trim.')


Ngày parse đủ, không NaT; giờ/phút/giây/micro đều 0 → ISO ngày không mất thông tin.
4 cột số/cờ: đúng int64, không null, phần thập phân bằng 0.
Chuỗi/mã giữ nguyên giá trị, không trim.


**Giải thích:** kiểm hợp lệ thật thay vì chỉ nhìn kiểu: số nguyên đúng dtype int64 và không phần thập phân; ngày không NaT và bằng phần ngày chuẩn hóa (kể cả giây nhỏ).

In [4]:
sach = nguon.copy()
sach['Sales_Date'] = ngay_ban.dt.strftime('%Y-%m-%d')
sach['Delivery_Date'] = ngay_giao.dt.strftime('%Y-%m-%d')
assert list(sach.columns) == COLS + TRACE
assert len(sach) == 9705 and int(sach[COLS].isna().sum().sum()) == 0
assert (pd.to_datetime(sach.Sales_Date) == ngay_ban).all()
assert (pd.to_datetime(sach.Delivery_Date) == ngay_giao).all()
for ten_cot in [c for c in COLS if sach[c].dtype == object]:
    assert (sach[ten_cot] == nguon[ten_cot]).all(), f'DUNG chuoi {ten_cot}'
for ten_cot in ['Qty', 'Unit_Price', 'Line_Amount', 'Is_Returned']:
    assert (sach[ten_cot] == nguon[ten_cot]).all()
print('Bảng sạch: chỉ khác định dạng ngày ISO; từng ô còn lại giữ nguyên.')


Bảng sạch: chỉ khác định dạng ngày ISO; từng ô còn lại giữ nguyên.


**Giải thích:** chuẩn hóa hình thức, kiểm từng ô theo mã truy nguồn.

In [5]:
dem_khach = sach.groupby('Order_ID').Customer_ID.nunique()
ma_can_xem = dem_khach[dem_khach > 1].index
bang_co = sach[sach.Order_ID.isin(ma_can_xem)].copy()
bang_co['n_customers'] = bang_co.Order_ID.map(dem_khach)
bang_co['reason'] = 'ORDER_MULTIPLE_CUSTOMER_IDS'
bang_co = bang_co[['Order_ID', 'Customer_ID'] + TRACE + ['n_customers', 'reason']].reset_index(drop=True)
assert len(ma_can_xem) == 3459 and len(bang_co) == 6918
khoa_co = bang_co['__source_file__'] + '|' + bang_co['__sheet__'] + '|' + bang_co['__excel_row__'].astype(str)
assert khoa_co.is_unique
print('Cờ đa khách: 3.459 nhóm / 6.918 dòng; khóa duy nhất; chỉ để xem xét.')
OUT.mkdir(parents=True, exist_ok=True)
sach.to_csv(CLEAN, index=False, encoding='utf-8')
bang_co.to_csv(FLAGS, index=False, encoding='utf-8')
print('Đã ghi 2 CSV UTF-8, không index thừa.')


Cờ đa khách: 3.459 nhóm / 6.918 dòng; khóa duy nhất; chỉ để xem xét.
Đã ghi 2 CSV UTF-8, không index thừa.


**Giải thích:** bảng cờ giữ mã đơn/khách, truy nguồn, số mã khách và lý do rõ. Không dùng loại dòng.

In [6]:
kieu_doc = {ten_cot: str for ten_cot in COLS if ten_cot not in ['Qty', 'Unit_Price', 'Line_Amount', 'Is_Returned']}
kieu_doc.update({ten_cot: 'int64' for ten_cot in ['Qty', 'Unit_Price', 'Line_Amount', 'Is_Returned']})
kieu_doc.update({'__source_file__': str, '__sheet__': str, '__excel_row__': 'int64'})
doc_lai = pd.read_csv(CLEAN, encoding='utf-8', dtype=kieu_doc)
co_lai = pd.read_csv(FLAGS, encoding='utf-8', dtype={'Order_ID': str, 'Customer_ID': str, '__source_file__': str, '__sheet__': str, '__excel_row__': 'int64', 'n_customers': 'int64', 'reason': str})
assert list(doc_lai.columns) == list(sach.columns) and len(doc_lai) == 9705
assert list(co_lai.columns) == list(bang_co.columns) and len(co_lai) == 6918
print('Đọc lại đúng schema/dòng, dtype rõ (ngày str ISO, số int64).')


Đọc lại đúng schema/dòng, dtype rõ (ngày str ISO, số int64).


**Giải thích:** đọc lại CSV từ đĩa với kiểu định rõ.

In [7]:
assert (doc_lai.astype(str) == sach.astype(str)).all().all(), 'DUNG: CSV sach doc lai khac RAM'
assert (co_lai.astype(str) == bang_co.astype(str)).all().all(), 'DUNG: CSV co doc lai khac'
khoa_lai = doc_lai['__source_file__'] + '|' + doc_lai['__sheet__'] + '|' + doc_lai['__excel_row__'].astype(str)
assert khoa_lai.is_unique
print('CSV đọc lại khớp từng ô với RAM; khóa truy nguồn duy nhất.')


CSV đọc lại khớp từng ô với RAM; khóa truy nguồn duy nhất.


**Giải thích:** đối chiếu ô giữa file đĩa và bảng trong RAM qua mã truy nguồn.

In [8]:
wb_rieng = openpyxl.load_workbook(DATA, read_only=True, data_only=True)
o_theo_ma = {}
for ten_sheet in wb_rieng.sheetnames:
    cac_dong = list(wb_rieng[ten_sheet].iter_rows(values_only=True))
    vi_tri = {ten_cot: i for i, ten_cot in enumerate(cac_dong[0])}
    for chi_so, dong_o in enumerate(cac_dong[1:], start=2):
        if all(o_goc is None for o_goc in dong_o): continue
        ma = f'data/Nile_Sales_YTD.xlsx|{ten_sheet}|{chi_so}'
        gia_tri = []
        for ten_cot in COLS:
            o_goc = dong_o[vi_tri[ten_cot]]
            if ten_cot in ('Sales_Date', 'Delivery_Date'):
                gia_tri.append(o_goc.strftime('%Y-%m-%d'))
            else:
                gia_tri.append(o_goc)
        o_theo_ma[ma] = gia_tri
wb_rieng.close()
so_lech = 0
for _, dong_csv in doc_lai.iterrows():
    ma = f"{dong_csv['__source_file__']}|{dong_csv['__sheet__']}|{dong_csv['__excel_row__']}"
    goc = o_theo_ma[ma]
    for ten_cot, o_goc in zip(COLS, goc):
        if str(dong_csv[ten_cot]) != str(o_goc):
            so_lech += 1
print('Ô CSV khác ô Excel gốc:', so_lech, '/ 174.690')
assert so_lech == 0
print('Từng giá trị trong 18 cột khớp Excel độc lập qua mã truy nguồn.')


Ô CSV khác ô Excel gốc: 0 / 174.690
Từng giá trị trong 18 cột khớp Excel độc lập qua mã truy nguồn.


**Giải thích:** đọc Excel lần hai hoàn toàn riêng (openpyxl, theo chỉ số cột), so từng ô CSV với ô gốc qua mã truy nguồn: 174.690 ô đều khớp.

In [9]:
thang_pd = doc_lai.assign(thang=pd.to_datetime(doc_lai.Sales_Date).dt.strftime('%Y-%m')).groupby('thang').agg(dong=('Qty', 'size'), luong=('Qty', 'sum'), tri=('Line_Amount', 'sum'))
nhom_pd = doc_lai.groupby('Category').agg(dong=('Qty', 'size'), luong=('Qty', 'sum'), tri=('Line_Amount', 'sum'))
noi_pd = doc_lai.groupby('To_Location').agg(dong=('Qty', 'size'), luong=('Qty', 'sum'), tri=('Line_Amount', 'sum'))
rieng = openpyxl.load_workbook(DATA, read_only=True, data_only=True)
gom_thang, gom_nhom, gom_noi = {}, {}, {}
for ten_sheet in rieng.sheetnames:
    cac_dong = list(rieng[ten_sheet].iter_rows(values_only=True))
    vi_tri = {ten_cot: i for i, ten_cot in enumerate(cac_dong[0])}
    for dong_o in cac_dong[1:]:
        if all(o_goc is None for o_goc in dong_o): continue
        ngay = dong_o[vi_tri['Sales_Date']]
        thang = f'{ngay.year}-{ngay.month:02d}'
        so_luong_o, tri_o = dong_o[vi_tri['Qty']], dong_o[vi_tri['Line_Amount']]
        for bang, khoa in [(gom_thang, thang), (gom_nhom, dong_o[vi_tri['Category']]), (gom_noi, dong_o[vi_tri['To_Location']])]:
            tich = bang.setdefault(khoa, [0, 0, 0])
            tich[0] += 1
            tich[1] += so_luong_o
            tich[2] += tri_o
rieng.close()
nhom_lech = []
for ten_loai, doc_lap, khung in [('thang', gom_thang, thang_pd), ('nhom', gom_nhom, nhom_pd), ('noi', gom_noi, noi_pd)]:
    assert set(doc_lap) == set(khung.index), f'DUNG tap khoa {ten_loai}'
    for khoa in doc_lap:
        that = (int(khung.loc[khoa, 'dong']), int(khung.loc[khoa, 'luong']), int(khung.loc[khoa, 'tri']))
        if tuple(doc_lap[khoa]) != that:
            nhom_lech.append((ten_loai, khoa))
print('Nhóm lệch dòng/Qty/trị:', nhom_lech or 'không có', '| tháng/Category/nơi:', len(gom_thang), len(gom_nhom), len(gom_noi))
assert not nhom_lech


Nhóm lệch dòng/Qty/trị: không có | tháng/Category/nơi: 7 5 28


**Giải thích:** đối chiếu dòng/Qty/trị theo 7 tháng, 5 Category, 28 nơi nhận giữa CSV và Excel độc lập, gồm cả tập khóa.

In [10]:
ban_thu = sach.copy()
ban_thu.loc[0, 'Qty'] += 1
ban_thu2 = sach.drop(index=0).reset_index(drop=True)
bat_qty = int(ban_thu.Qty.sum()) != int(sach.Qty.sum())
mat_dau_vet = len(ban_thu2) != len(sach)
print('Sửa Qty+1 bị bắt:', bat_qty, '| mất 1 dấu vết bị bắt:', mat_dau_vet)
assert bat_qty and mat_dau_vet
assert len(sach) == 9705 and int(sach.Qty.sum()) == 13087
print('Bảng thật nguyên vẹn; kiểm đủ nhạy. Không sửa nguồn/CSV khi thử.')


Sửa Qty+1 bị bắt: True | mất 1 dấu vết bị bắt: True
Bảng thật nguyên vẹn; kiểm đủ nhạy. Không sửa nguồn/CSV khi thử.


**Giải thích:** kiểm nhỏ trên bản sao RAM: sai Qty+1 và mất dấu vết đều bị phát hiện.

In [11]:
loi = []
def chot(ten, ky_vong, thuc_te):
    dat = ky_vong == thuc_te
    print(f'{ten}: kỳ vọng {ky_vong} | thực tế {thuc_te} → {"ĐẠT" if dat else "KHÔNG ĐẠT"}')
    if not dat:
        loi.append(ten)
chot('hash đầu', 'fae7a6955da3f3166ef4766d26acc39b8fed7b8c763e077a83c015a6d4fc0bd8'[:12], sha_dau[:12])
chot('dòng', 9705, len(doc_lai))
chot('Qty/trị', (13087, 8787091), (int(doc_lai.Qty.sum()), int(doc_lai.Line_Amount.sum())))
chot('mã đơn/đa khách', (6246, 3459), (int(doc_lai.Order_ID.nunique()), int((doc_lai.groupby('Order_ID').Customer_ID.nunique() > 1).sum())))
chot('cờ 1/mẫu', (574, 9705), (int((doc_lai.Is_Returned == 1).sum()), int(doc_lai.Is_Returned.isin([0, 1]).sum())))
chot('dòng cờ', 6918, len(co_lai))
assert not loi, f'DỪNG: {loi}'
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == sha_dau == 'fae7a6955da3f3166ef4766d26acc39b8fed7b8c763e077a83c015a6d4fc0bd8'
print('Tất cả ĐẠT; nguồn không đổi suốt run.')


hash đầu: kỳ vọng fae7a6955da3 | thực tế fae7a6955da3 → ĐẠT
dòng: kỳ vọng 9705 | thực tế 9705 → ĐẠT
Qty/trị: kỳ vọng (13087, 8787091) | thực tế (13087, 8787091) → ĐẠT
mã đơn/đa khách: kỳ vọng (6246, 3459) | thực tế (6246, 3459) → ĐẠT
cờ 1/mẫu: kỳ vọng (574, 9705) | thực tế (574, 9705) → ĐẠT
dòng cờ: kỳ vọng 6918 | thực tế 6918 → ĐẠT
Tất cả ĐẠT; nguồn không đổi suốt run.


**Giải thích:** bảng chốt fail thật khi lệch. Cờ đa khách là điều cần xác minh theo H1, không phải lỗi; không dùng bảng cờ loại dòng.